# Analyze mapQC output Santamaria mapping

In [ ]:
import scanpy as sc
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors

sys.path.append("../../")
sys.path.append("../utils/HECA")

import mapqc
import data_cleanup

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
path_results_dir = "/lustre/groups/ml01/code/lisa.sikkema/mapqc_reproducibility/results/HECA/batch_sample_kl1/mapqc"
path_figure_dir = "/lustre/groups/ml01/code/lisa.sikkema/mapqc_figures/heca_mapping/"
path_table_dir = "/lustre/groups/ml01/code/lisa.sikkema/mapqc_tables/heca_mapping/"
path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria_batch_sample_kl1_latent.h5ad"
path_counts = (
    "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria.h5ad"
)

load anndata objects:

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)

In [ ]:
adata = data_cleanup.add_mapping_info_to_counts_adata(
    adata_counts, adata_emb, celltype_col="celltype"
)

load mapqc results:

In [ ]:
results_healthy = mapqc.load(
    os.path.join(path_results_dir, f"control_edistance_k500.pkl")
)
results_disease = mapqc.load(os.path.join(path_results_dir, f"case_edistance_k500.pkl"))

## Analyze

print results summary:

In [ ]:
results_healthy.result_stats

In [ ]:
results_disease.result_stats

#### Some data cleanup:

add some ct info and order/clean up:

In [ ]:
for col in ["celltype", "celltype_merged", "celltype_query"]:
    adata.obs[col] = [
        f"{label[0].upper()}{label[1:]}" if isinstance(label, str) else label
        for label in adata.obs[col]
    ]

In [ ]:
ct_to_lineage_df = adata.obs.groupby("celltype").agg({"lineage": "first"})
ct_to_lineage_df["celltype"] = ct_to_lineage_df.index
ct_to_lineage_df.index.name = None
ct_to_lineage_df.sort_values(by=["lineage", "celltype"], inplace=True)
ct_to_lineage_mapper = {
    ct: lin for ct, lin in zip(ct_to_lineage_df.index, ct_to_lineage_df.lineage)
}
adata.obs["lineage_merged"] = adata.obs.celltype_merged.map(ct_to_lineage_mapper)

In [ ]:
adata.obs["celltype_merged"] = pd.Categorical(
    adata.obs.celltype_merged,
    categories=ct_to_lineage_df.celltype.tolist(),
    ordered=False,
)

In [ ]:
adata.obs["donor_query"] = [
    None
    if r_or_q == "r"
    else (
        s.split("-")[-3]
        if (group == "WOI Control") or ("biop" in s)
        else s.split("-")[-2]
    )
    for r_or_q, s, group in zip(adata.obs.r_or_q, adata.obs["sample"], adata.obs.Group)
]

copy mapqc scores from mapqc results object to adata:

In [ ]:
q_ctr_cells = results_healthy.adata.obs.loc[
    results_healthy.adata.obs.r_or_q == "q", :
].index
q_case_cells = results_disease.adata.obs.loc[
    results_disease.adata.obs.r_or_q == "q", :
].index
for col in ["mapqc_score", "mapqc_score>2"]:
    adata.obs[col] = None
    adata.obs[col] = None
    adata.obs.loc[q_ctr_cells, col] = results_healthy.adata.obs.loc[q_ctr_cells, col]
    adata.obs.loc[q_case_cells, col] = results_disease.adata.obs.loc[q_case_cells, col]

simplify "group" category (different types of controls, plus disease category) to two-category covariate:

In [ ]:
adata.obs["status"] = adata.obs.Group.map(
    {
        "Ctrl_Superficial": "control",
        "Ctrl_whole_uterus": "control",
        "WOI Control": "control",
        "AS": "Asherman's Syndrome",
    }
)

create column specifying whether a cell is from the reference or the query, and then specifying whether it is control or case:

In [ ]:
# create variable splitting data in ref control, query control
adata.obs["r_q_status"] = [
    f"{r_q} {stat}" for r_q, stat in zip(adata.obs.r_or_q, adata.obs.status)
]

## Plot figures:

In [ ]:
sc.set_figure_params(figsize=(5, 5), dpi=100)
s = 0.5

In [ ]:
# create empty dict for storing figures for paper:
FIGURES = dict()
TABLES = dict()

custom color map:

In [ ]:
cmap = {
    "r control": "grey",
    "q control": "dodgerblue",
    "q Asherman's Syndrome": "midnightblue",
}

In [ ]:
FIGURES["umap_r_or_q_ctr_or_case"] = sc.pl.umap(
    adata,
    color="r_q_status",
    palette=cmap,
    frameon=False,
    show=False,
    return_fig=True,
    s=s,
)

### Results healthy

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, color_by="mean_pval_corr", size=0.5)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "dataset",
        "celltype_merged",
        "lineage_merged",
        "celltype_query",
    ],
    ncols=2,
    wspace=0.75,
    frameon=False,
)

In [ ]:
FIGURES["umap_ctr_and_case_mapqc_score_binarized"] = sc.pl.umap(
    adata,
    color="mapqc_score>2",
    palette={
        ">2": "firebrick",
        "<=2": "navajowhite",
        "filtered out": "black",
        "not sampled": "cornflowerblue",
    },
    frameon=False,
    na_color="gray",
    size=s,
    show=False,
    return_fig=True,
)

In [ ]:
mapqc.pl.umap_mapqc_scores(
    results_healthy, color_by="mean_pval_corr_significant", size=0.5
)

In [ ]:
FIGURES["umap_ctr_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_healthy, color_by="mapqc_score>2", size=s, figsize=(7, 7), return_fig=True
)

#### intermezzo: look into specific neighborhood

In [ ]:
nhood_oi = results_healthy.nhood_stats.index[
    results_healthy.nhood_stats.nhood_number == 734
][0]

In [ ]:
results_healthy.nhood_stats.loc[nhood_oi, :]

In [ ]:
mapqc.pl.umap_center_cell(results_healthy, neighborhood_center_cell=nhood_oi)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(results_healthy, neighborhood=nhood_oi)

In [ ]:
mapqc.pl.umap_show_neighborhood(
    results_healthy,
    neighborhood=nhood_oi,
    color_by="dataset",
)

In [ ]:
mapqc.pl.umap_q_ref_sample_shaded(results_healthy, neighborhood=nhood_oi, dotsize=10)

In [ ]:
mapqc.pl.umap_neighborhood(
    results_healthy,
    color_by=["sample", "dataset", "mapqc_score", "mapqc_score>2"],
    neighborhood=nhood_oi,
    recalculate_umap=False,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
)

In [ ]:
mapqc.pl.umap_neighborhood(
    results_healthy,
    color_by=["sample", "dataset", "mapqc_score>2"],
    neighborhood=nhood_oi,
    recalculate_umap=True,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
    size=20,
)

#### end of intermezzo

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, size=0.5)

In [ ]:
mapqc.pl.umap_filtering_info(results_healthy)

color by query sample:

In [ ]:
adata.obs["sample_id_q"] = None
adata.obs.loc[adata.obs.r_or_q == "q", "sample_id_q"] = adata.obs.loc[
    adata.obs.r_or_q == "q", "sample"
].tolist()

In [ ]:
sc.pl.umap(
    adata,
    color=["dataset", "sample_id_q", "celltype_merged"],
    frameon=False,
    wspace=0.7,
    ncols=2,
)

In [ ]:
palette_boxplots = {  # need to use a different palette here, as with query dark blue we cannot see individual points on the boxes anymore
    "WOI Control": "#75bbfd",
    "AS": matplotlib.colors.to_rgba("#276ab3"),
}

check how many outliers we exclude by setting vmax to 20:

In [ ]:
(results_healthy.adata.obs.mapqc_score > 20).sum()

In [ ]:
(results_disease.adata.obs.mapqc_score > 20).sum()

In [ ]:
FIGURES[
    "grouped_boxplot_leiden_mapqc_score"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="leiden",
    group_category="Group",
    palette=palette_boxplots,
    figsize=(10, 4),
    min_cells_per_box=50,
    vmax=20,
    return_fig=True,
)

plot umap showing location of two clusters with highest mapQC scores:

In [ ]:
sc.pl.umap(
    adata, color="leiden", groups=["21", "25"], frameon=False, palette="Dark2", s=1
)

show number of reference and query cells in top 3  clusters:

In [ ]:
pd.crosstab(adata.obs.leiden, adata.obs.r_or_q).loc[["3", "21", "25"], :]

show mapqc score per query annotation (annotations from original query publication)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="celltype_query",
    group_category="Group",
    palette=palette_boxplots,
    min_cells_per_box=100,
    vmax=10,
)

show query celltype that was disease specific according to publication (with highest mapqc scores of all query annotations, see above)

In [ ]:
sc.pl.umap(
    adata,
    color="celltype_query",
    groups=["AS-Epithelium"],
    ncols=2,
    frameon=False,
)

now show mapQC scores per annotation as transferred from the reference. This is how the normal analysis pipeline would be, as normally we do not have annotations for a new query dataset:

In [ ]:
FIGURES[
    "grouped_boxplot_ann_mapqc_score"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="celltype_pred",
    group_category="Group",
    palette=palette_boxplots,
    min_cells_per_box=50,
    vmax=20,
    figsize=(10, 6),
    return_fig=True,
)

### Results disease

In [ ]:
mapqc.pl.umap_mapqc_scores(results_disease, size=1)

In [ ]:
sc.pl.umap(adata, color="leiden", legend_loc="on data")

In [ ]:
mapqc.pl.umap_mapqc_scores(results_disease, color_by="mapqc_score>2", size=1)

In [ ]:
mapqc.pl.umap_mapqc_scores(
    results_disease, color_by="mean_pval_corr_significant", size=1
)

In [ ]:
mapqc.pl.umap_filtering_info(results_disease)

In [ ]:
results_disease.adata.obs["donor_query"] = adata.obs.loc[
    results_disease.adata.obs.index, "donor_query"
]

show mapqc results per ct or cluster, coloring by query donor, just to check if specific cts/clusters are mostly enriched for a single donor (answer is yes, in the case of clusters and original query ct annotations)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="celltype_pred", color_category="donor_query"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="leiden", color_category="donor_query"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="celltype_query", color_category="donor_query"
)

Out of curiosity: which transferred annotations did cells originally annotated as AS-epithelium get?

In [ ]:
pd.crosstab(adata.obs.celltype_query, adata.obs.celltype_pred).loc[
    "AS-Epithelium", :
].sort_values(ascending=False)[:10]

#### Gene-level analysis

Note that quite some genes were only detected in the query, these are useless for analysis as they are batch-driven. We will therefore subset the anndata for common genes for any dea analysis.

In [ ]:
ref_gene_sums = np.array(adata[adata.obs.r_or_q == "r", :].X.sum(axis=0))
genes_not_in_reference = adata.var.index[ref_gene_sums[0] == 0]
adata_common_genes_only = adata[:, ~adata.var.index.isin(genes_not_in_reference)].copy()

In [ ]:
# adata.obs["celltype_cond_mapqc"] = [
#     f"{ct} ({cond}, {mapqc})"
#     if cond in ["q Asherman's Syndrome", "q control"]
#     else f"{ct} (ref)"
#     for ct, cond, mapqc in zip(
#         adata.obs.celltype_merged, adata.obs.r_q_status, adata.obs["mapqc_score>2"]
#     )
# ]
# adata.obs["celltype_cond_mapqc"] = [
#     f"{ct_cond} , mapqc {mapqc})"
#     for ct_cond, mapqc in zip(adata.obs.celltype_cond, adata.obs["mapqc_score>2"])
# ]

check in which cluster the AS epithelium is largely located:

In [ ]:
print(
    "Number of AS-epithelium cells:",
    adata.obs.celltype_query.value_counts().loc["AS-Epithelium"],
)

In [ ]:
n_cells_cl21 = adata.obs.leiden.value_counts().loc["21"]
print("Number of cells in cluster 21:", n_cells_cl21)

In [ ]:
adata.n_obs - n_cells_cl21

show number of AS-epithelium cells per cluster:

In [ ]:
pd.crosstab(
    adata.obs.celltype_query,
    adata.obs.leiden,
).loc[
    "AS-Epithelium"
].sort_values(ascending=False)[:10]

Do the same for other putative disease-related cell type, Epi-PDGFRA:

In [ ]:
pd.crosstab(
    adata.obs.celltype_query,
    adata.obs.leiden,
).loc["Epi-PDGFRA"].sort_values(
    ascending=False
)[:10]

mapqc scores per annotation:

In [ ]:
pd.crosstab(
    adata.obs.celltype_query,
    adata.obs["mapqc_score>2"],
).loc["AS-Epithelium"]

In [ ]:
pd.crosstab(
    adata.obs.celltype_query,
    adata.obs["mapqc_score>2"],
).loc["Epi-PDGFRA"]

number of query control versus query disease cells per annotation:

In [ ]:
pd.crosstab(
    adata.obs.celltype_query,
    adata.obs.r_q_status,
).loc["AS-Epithelium"]

In [ ]:
pd.crosstab(
    adata.obs.celltype_query,
    adata.obs.r_q_status,
).loc["Epi-PDGFRA"]

Cluster 21 or 25, number of disease versus control cells (query) and number of high versus low mapqc cells:

In [ ]:
pd.crosstab(
    adata.obs.loc[adata.obs.leiden == "25", "Group"],
    adata.obs.loc[adata.obs.leiden == "25", "mapqc_score>2"],
)

In [ ]:
pd.crosstab(
    adata.obs.loc[adata.obs.leiden == "21", "Group"],
    adata.obs.loc[adata.obs.leiden == "21", "mapqc_score>2"],
)

check which reference-based annotations are in cl 21:

In [ ]:
adata.obs.loc[adata.obs.leiden == "21", "celltype_merged"].value_counts()[:10]

Calculate degs for cluster 21. Note: this gives a mix of epithelial and stromal markers, as this cluster has both markers (seems like it includes doublets)

In [ ]:
sc.tl.rank_genes_groups(
    adata_common_genes_only,
    groupby="leiden",
    key_added="dea_leiden",
    groups=["25", "21"],
    reference="rest",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_common_genes_only, key="dea_leiden", key_added="dea_leiden_filtered"
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_common_genes_only,
    key="dea_leiden",
    key_added="dea_leiden_filtered_strict",
    max_out_group_fraction=0.2,
    min_in_group_fraction=0.5,
)

In [ ]:
deg_21 = sc.get.rank_genes_groups_df(
    adata_common_genes_only, group="21", key="dea_leiden"
)
deg_21_filtered = sc.get.rank_genes_groups_df(
    adata_common_genes_only, group="21", key="dea_leiden_filtered"
).dropna(how="any")
deg_21["passed_filtering"] = deg_21.names.isin(deg_21_filtered.names)

In [ ]:
TABLES["cluster_21_versus_rest"] = deg_21

In [ ]:
deg_21_filtered.shape

In [ ]:
deg_25 = sc.get.rank_genes_groups_df(
    adata_common_genes_only, group="25", key="dea_leiden"
)
deg_25_filtered = sc.get.rank_genes_groups_df(
    adata_common_genes_only, group="25", key="dea_leiden_filtered"
).dropna(how="any")

plots for cluster 21. Note that genes upregulated in 21 are epithelial/stromal markers, whereas genes downregulated are genes generally lowly measured in the query.

create obs column with celltype merged except for cluster 21

In [ ]:
adata.obs["leiden_21_or_ct"] = [
    leiden if leiden == "21" else ct
    for leiden, ct in zip(adata.obs.leiden, adata.obs.celltype_merged)
]

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=deg_21_filtered.names,
    groupby="leiden_21_or_ct",
    standard_scale="var",
)

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=deg_21_filtered.names[:20],
    groupby="lineage",
    standard_scale="var",
)

create column with lineage annotation, or cluster 21 subset:

In [ ]:
adata.obs["leiden_21_or_ct_r_or_q"] = [
    f"{leiden} ({status})" if leiden == "21" else f"{ct} ({status})"
    for leiden, ct, status in zip(
        adata.obs.leiden, adata.obs.celltype_merged, adata.obs.r_q_status
    )
]

In [ ]:
adata.obs["lineage_or_leiden_21_group"] = [
    leiden_subgroup if leiden_subgroup.startswith("21") else lin
    for leiden_subgroup, lin in zip(
        adata.obs.leiden_21_or_ct_r_or_q, adata.obs.lineage_merged
    )
]

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=deg_21_filtered.names[:20],
    groupby="lineage_or_leiden_21_group",
    standard_scale="var",
)

store clean DEGs from cluster 21, and specify if they are epithelial or stromal 

In [ ]:
deg_21_filtered_ordered = {
    "Epithelial": ["SCGB2A1", "SCGB1D2", "WFDC2", "SCGB1D4", "ASRGL1", "ENPP3"],
    "Stromal": ["PTN", "MME"],
}

plot for all cell types (plus cluster 21):

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=deg_21_filtered_ordered,
    groupby="leiden_21_or_ct",
    standard_scale="var",
)

check downregulated genes in cluster 21 compared to rest *of query*, only look at genes also present in the reference, so that we can fairly compare to reference later:

In [ ]:
adata_query = adata_common_genes_only[
    adata_common_genes_only.obs.r_or_q == "q", :
].copy()

In [ ]:
adata_query.obs["21_or_other"] = [
    "21" if cl == "21" else "other" for cl in adata_query.obs.leiden
]

In [ ]:
adata_query.obs["21_or_other"].value_counts()

In [ ]:
sc.tl.rank_genes_groups(
    adata_query,
    groupby="21_or_other",
    key_added="rest_vs_21",
    groups=["other"],
    reference="21",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_query,
    key="rest_vs_21",
    key_added="rest_vs_21_filt",
    min_in_group_fraction=0.5,
)

In [ ]:
deg_21_down_query_only_unfiltered = sc.get.rank_genes_groups_df(
    adata_query, group="other", key="rest_vs_21"
)

In [ ]:
deg_21_down_query_only = sc.get.rank_genes_groups_df(
    adata_query, group="other", key="rest_vs_21_filt"
).dropna(how="any")

In [ ]:
deg_21_down_query_only_unfiltered[
    "passed_filtering"
] = deg_21_down_query_only_unfiltered.names.isin(deg_21_down_query_only.names)

In [ ]:
TABLES["query_rest_vs_query_cluster_21"] = deg_21_down_query_only_unfiltered

In [ ]:
deg_21_down_query_only.shape

create column with leiden cluster + r or q (cond):

In [ ]:
adata.obs["leiden_21_r_or_q_cond"] = [
    f"{leiden} ({cond})" if ((r_or_q == "q") and (leiden == "21")) else leiden
    for leiden, r_or_q, cond in zip(
        adata.obs.leiden, adata.obs.r_or_q, adata.obs.r_q_status
    )
]
adata.obs.leiden_r_or_q_cond = data_cleanup.clean_up_df_column(
    adata.obs, "leiden_21_r_or_q_cond", min_n_cells_per_cat=50
)

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=deg_21_filtered.names[:20].tolist()
    + ["SLPI", "HSPA1A", "SOCS3", "MMP7", "SNHG9"],
    groupby="leiden_21_r_or_q_cond",
)

#### plots for cl 25:

query ct annotations in cluster 25:

In [ ]:
adata.obs.loc[adata.obs.leiden == "25", "celltype_query"].value_counts()[:5]

cl 25 mapqc scores:

In [ ]:
adata.obs.loc[adata.obs.leiden == "25", "mapqc_score>2"].value_counts()

conditions in cl 25:

In [ ]:
adata.obs.loc[adata.obs.leiden == "25", "Group"].value_counts()

degs versus rest are simply epithelial/lineage markers

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=deg_25_filtered.names[:20],
    groupby="lineage",
    standard_scale="var",
)

## Figure drafts for paper:

(for mapqc score plots, see above)

#### Numbers:

Number of reference and query cells:

In [ ]:
adata.obs.r_or_q.value_counts()

Check number of samples and donors:

In [ ]:
query_sample_info = (
    adata.obs.loc[adata.obs.r_or_q == "q", :]
    .groupby("sample")
    .agg({"Group": "first", "donor_query": "first"})
).dropna(how="any")

In [ ]:
query_donor_info = adata.obs.groupby("donor_query").agg(
    {"sample": "nunique", "Group": "first"}
)

In [ ]:
query_donor_info.Group.value_counts()

In [ ]:
perc_as_epi_in_cl_21 = (
    ((adata.obs.leiden == "21") & (adata.obs.celltype_query == "AS-Epithelium")).sum()
    / (adata.obs.celltype_query == "AS-Epithelium").sum()
    * 100
)
print(
    "Percentage of total AS epithelium in cluster 21:", round(perc_as_epi_in_cl_21, 1)
)

In [ ]:
perc_q_in_cl_21 = (
    ((adata.obs.leiden == "21") & (adata.obs.r_or_q == "q")).sum()
    / (adata.obs.leiden == "21").sum()
    * 100
)
perc_q_in_cl_25 = (
    ((adata.obs.leiden == "25") & (adata.obs.r_or_q == "q")).sum()
    / (adata.obs.leiden == "25").sum()
    * 100
)
print(
    "Percentage of cells in cluster 21/25 resp. from query:",
    round(perc_q_in_cl_21, 1),
    round(perc_q_in_cl_25, 1),
)

In [ ]:
adata.obs["21_25_or_rest"] = [
    leiden if leiden in ["21", "25"] else "Other clusters"
    for leiden in adata.obs.leiden
]

In [ ]:
cmap = {
    "r control": "gray",
    "q control": "dodgerblue",
    "q Asherman's Syndrome": "midnightblue",
}

In [ ]:
cmap = {
    "r control": "gray",
    "q control": "dodgerblue",
    "q Asherman's Syndrome": "midnightblue",
}
group_proportions_21_25_or_other = pd.crosstab(
    adata.obs["21_25_or_rest"], adata.obs.r_q_status
)
group_proportions_21_25_or_other_norm = group_proportions_21_25_or_other.divide(
    group_proportions_21_25_or_other.sum(axis=1), axis=0
)
fig, ax = plt.subplots(figsize=(2, 4))
group_proportions_21_25_or_other_norm.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[cmap[col] for col in group_proportions_21_25_or_other_norm.columns],
)
plt.grid(False)
# legend reverse order:
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc=(1.01, 0.7), frameon=False)
ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Cluster")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

show mapqc scores per subgroup:

In [ ]:
adata.obs["leiden_21_25_r_or_q_or_ct"] = [
    f"{leiden} ({status})" if leiden in ["21", "25"] else ct
    for leiden, status, ct in zip(
        adata.obs.leiden, adata.obs.r_q_status, adata.obs.celltype_merged
    )
]

In [ ]:
boxplot_df = adata.obs.loc[
    adata.obs.leiden.isin(["21", "25"]), ["leiden", "r_q_status", "mapqc_score"]
]
boxplot_df.leiden = boxplot_df.leiden.tolist()

In [ ]:
adata.obs["total_counts"] = adata.layers["counts"].sum(axis=1)
adata.obs["log10_total_counts"] = np.log10(adata.obs.total_counts)

show total counts for query only (as we want to understand the quality of query cells in these clusters as compared to the rest of the query), and only for boxes with at least 50 cells:

In [ ]:
query_obs = adata.obs.loc[adata.obs.r_or_q == "q", :].copy()

In [ ]:
cluster_counts = query_obs.leiden.value_counts()
clusters_to_keep = cluster_counts.loc[cluster_counts >= 50].index

In [ ]:
FIGURES["boxplot_query_counts_per_cell"], ax = plt.subplots(figsize=(6, 1))
sns.boxplot(
    query_obs.loc[query_obs.leiden.isin(clusters_to_keep), :],
    x="leiden",
    y="log10_total_counts",
    ax=ax,
    fliersize=0.1,
    color="gray",
)
ax.tick_params(axis="x", rotation=90)
ax.set_xticklabels(ax.get_xticks(), size=9)
ax.set_yticklabels(ax.get_yticks(), size=9)
ax.set_xlabel("Cluster", fontsize=10)
ax.set_ylabel("Log10(total counts)", fontsize=10)
plt.show()

MALAT1 expression per cluster:

In [ ]:
query_obs["MALAT1"] = adata[query_obs.index, "MALAT1"].X.toarray()

In [ ]:
FIGURES["boxplot_query_MALAT1_per_cell"], ax = plt.subplots(figsize=(6, 1))
sns.boxplot(
    query_obs.loc[query_obs.leiden.isin(clusters_to_keep), :],
    x="leiden",
    y="MALAT1",
    ax=ax,
    fliersize=0.1,
    color="gray",
)
ymin, ymax = ax.get_ylim()
yticks = np.arange(0, int(np.ceil(ymax)) + 1, 2)
ax.set_yticks(yticks)
ax.tick_params(axis="x", rotation=90)
ax.set_xticklabels(ax.get_xticks(), size=9)
ax.set_yticklabels(ax.get_yticks(), size=9)
ax.set_xlabel("Cluster", fontsize=10)
ax.set_ylabel("MALAT1", fontsize=10)
plt.show()

genes based on DEA, commenting out the ones that are not ubiquitous but only occur in a subset of the cell types based on visual plot inspection:

In [ ]:
deg_21_down_ubiquitous = [
    "MT-ATP8",
    "MBNL1",
    "RNF213",
    "PRRC2C",
    "ARID1B",
    "SF1",
    # 'SLC2A3',
    # 'LRP1',
    "JMJD1C",
    # 'ZEB2',
    "KANSL1",
    # 'CD248',
    "KMT2A",
    "RSBN1L",
    "HNRNPUL1",
    # 'CTSC',
    # 'SYNE1',
    # "TLE4",
    "ZBTB7A",
]

summarize the cluster 21 gene trends with three named groups:

In [ ]:
cl_21_genes = {
    "Stromal\n(up in 21)": ["PTN", "MME"],
    "Epithelial\n(up in 21 q. disease)": [
        "SCGB2A1",
        "SCGB1D2",
        "WFDC2",
        "SCGB1D4",
        "ASRGL1",
        "ENPP3",
    ],
    "Ubiquitous\n(down in cl. 21 query)": deg_21_down_ubiquitous,
}

also, specify cell type markers as based on the atlas paper:

In [ ]:
ct_markers_paper = {
    "Mesenchyme": ["DCN", "PCOLCE"],
    "EStromal": ["TUNAR", "F13A1", "TGFBI", "FOXL2", "CRABP2", "MMP11"],
    "EStromal MMPs": ["MMP10", "INHBA", "COL8A1", "MMP3", "MMP1"],
    "DStromal": ["DKK1", "LMCD1", "IL15", "C3", "PALMD", "CFD"],
    "DStromal early": ["PLCL1", "PCSK6", "TPPP3", "CILP", "CALB2"],
    "DStromal mid": ["TLR4", "CXCL13", "SCARA5", "GABRA2"],
    "DStromal late": ["CXCL2", "LEFTY2", "TRIB1", "CXCL8", "SMAD7"],
    "Fibroblast basalis": ["C7", "PTGIS"],
    "USMC": ["PTGER3", "DES", "ACTG2"],
    "PV general": ["RGS5"],
    "Epithelium": ["PAX8", "EPCAM"],
    "SOX9 basalis": ["CDH2"],
    "Epithelium prolif. phase": ["MMP7"],
    "SOX9 prolif. phase": ["KMO", "EMID1", "IHH"],
    "Preglandular": ["SUFU", "HPRT1", "OPRK1"],
    "Glandular secretory/\nSOX9 luminal": ["GPX3", "DPP4", "PAEP"],
    "(Pre)luminal": ["LGR5"],
    "(Pre)ciliated": ["TP73"],
    "MUC5B": ["MUC5B"],
    "Endothelial": ["CDH5"],
    "Immune": ["PTPRC"],
}

In [ ]:
adata.obs["lineage_r_or_q_or_leiden_group"] = [
    gr if gr.startswith("21") else f"{gr} ({r_or_q})"
    for gr, r_or_q in zip(adata.obs.lineage_or_leiden_21_group, adata.obs.r_or_q)
]

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=cl_21_genes,
    groupby="lineage_r_or_q_or_leiden_group",
    standard_scale="var",
)

In [ ]:
sc.pl.dotplot(
    adata,
    groupby="leiden_21_25_r_or_q_or_ct",
    var_names=ct_markers_paper,
    standard_scale="var",
)

## Store figures and tables

In [ ]:
path_figure_dir

In [ ]:
FIGURES

In [ ]:
for figname, fig in FIGURES.items():
    if "umap" not in figname:  # TEMP FOR FASTER SAVING, ALREADY HAVE ALL UMAPS
        print("Saving", figname)
        fig.savefig(
            os.path.join(path_figure_dir, f"{figname}.png"),
            bbox_inches="tight",
            dpi=140,
            transparent=True,
        )

In [ ]:
path_table_dir

In [ ]:
TABLES.keys()

In [ ]:
for table_name, table in TABLES.items():
    table.to_csv(os.path.join(path_table_dir, f"{table_name}.csv"), index=False)